In [5]:
import pandas as pd
import numpy as np
import requests
import time
import mygene

from Bio import Entrez, SeqIO

In [2]:
df_final = pd.read_csv("datosGene4PD/coordenadas_genes.csv", sep = ",", index_col = False)

In [3]:
df_final

,Chr,gene_symbol,inicio,SNP_position,fin,effect_allele,alternate_allele,cadena
0,6,GPR126,142622991,142758601,142767403,T,G,1.0
1,1,SYT11,155829300,155839054,155854990,C,T,1.0
2,12,SLC2A13,40148823,40428561,40499891,G,T,-1.0
3,12,SLC2A13,40148823,40478652,40499891,G,T,-1.0
4,12,SLC2A13,40148823,40474147,40499891,C,T,-1.0
...,...,...,...,...,...,...,...,...
628,17,BRIP1,59758627,59917366,59940882,T,C,-1.0
629,17,DNAH17,76419778,76425480,76573476,A,T,-1.0
630,18,ASXL3,31158579,31304318,31331156,T,G,1.0
631,20,CRLS1,5986736,6006041,6020699,T,C,1.0


In [8]:
lista_symbols = []

for i in range(len(df_final)):
    lista_symbols.append(df_final.iloc[i]["gene_symbol"])

In [6]:
Entrez.email = "jbs1009@alu.ubu.es"

def descargar_secuencia(gene_symbol):    
    
    query = f"{gene_symbol}[Gene Name] AND Homo sapiens[Organism] AND refseq[filter] AND biomol_mrna[prop]"
    
    try:
        
        handle = Entrez.esearch(db = "nucleotide", term = query, retmax = 1)
        record = Entrez.read(handle)
        handle.close()
        
        if not record["IdList"]:
            return None
        
        seq_id = record["IdList"][0]
        
        fetch_handle = Entrez.efetch(db = "nucleotide", id = seq_id, rettype = "fasta", retmode = "text")
        
        seq_record = SeqIO.read(fetch_handle, "fasta")
        fetch_handle.close()
        
        return str(seq_record.seq)
    
    except Exception as e:
        
        print(f"Error al descargar {gene_symbol}: {e}")
        return None

In [12]:
secuencias_guardadas = {}

for gen in lista_symbols:
    print(f"Buscando: {gen}", end = " ")
    
    secuencia = descargar_secuencia(gen)
    
    if secuencia:
        secuencias_guardadas[gen] = secuencia
        print("Encontrado.")
    else:
        print("No encontrado.")
    
    time.sleep(0.5)

Buscando: GPR126 Encontrado.
Buscando: SYT11 Encontrado.
Buscando: SLC2A13 Encontrado.
Buscando: SLC2A13 Encontrado.
Buscando: SLC2A13 Encontrado.
Buscando: LRRK2 Encontrado.
Buscando: LRRK2 Encontrado.
Buscando: SERPINA1 Encontrado.
Buscando: SERPINA1 Encontrado.
Buscando: MAPT Encontrado.
Buscando: SPPL2C Encontrado.
Buscando: HLA-DRA Encontrado.
Buscando: DGKQ Encontrado.
Buscando: NSF Encontrado.
Buscando: GAK Encontrado.
Buscando: WNT3 Encontrado.
Buscando: CSMD1 Encontrado.
Buscando: TAS2R19 Encontrado.
Buscando: WNT3 Encontrado.
Buscando: UNC13B Encontrado.
Buscando: LINC00693 No encontrado.
Buscando: MPHOSPH10 Encontrado.
Buscando: ZNF519 Encontrado.
Buscando: AAK1 Encontrado.
Buscando: CDH6 Encontrado.
Buscando: GRB10 Encontrado.
Buscando: PIK3CD Encontrado.
Buscando: SNCA Encontrado.
Buscando: PLA2R1 Encontrado.
Buscando: SH3GL2 Encontrado.
Buscando: PLA2R1 Encontrado.
Buscando: CAST Encontrado.
Buscando: SLCO3A1 Encontrado.
Buscando: SPPL2C Encontrado.
Buscando: DNAH11 Encon

In [15]:
len(secuencias_guardadas)

257

In [17]:
df_secs = pd.DataFrame(list(secuencias_guardadas.items()), columns = ["gene_symbol", "secuencia"])

In [18]:
df_secs

,gene_symbol,secuencia
0,GPR126,AGGAGTAACAGGCACCGCTCCTCAGTCCAGAGGCCTGGCCCTGCCA...
1,SYT11,AGCGCATCCCCGGAGCATCTTAAGAGCTGAGCGCAGCTGACAACTA...
2,SLC2A13,TGACAGACACACACATAAATGCACAAATAATTCTGGCTAACTTCAC...
3,LRRK2,GGGGCCCGCGGGGAGCGCTGGCTGCGGGCGGTGAGCTGAGCTCGCC...
4,SERPINA1,AGAGTCCTGAGCTGAACCAAGAAGGAGGAGGGGGTCGGGCCTCCGA...
...,...,...
252,BRIP1,CGCACGGCTTCTGGCGGCGCCAAACACCCGGTATTTATTTCGCCCC...
253,DNAH17,ATGTGGCAGTTGGAGCCCCTCGAGGGAAGGGAGCCATTTTCTTTCC...
254,ASXL3,AGGCACTAGAAAAACACCCCAACTCACCAATGACAGCAAAGCAGAT...
255,CRLS1,AGTATGGAGGCAGCGGTAGCCCAGTGTCTGAGTGGTTGCCGGGTCT...


In [19]:
df_secs.to_csv('datosGene4PD/secuencias_genes.csv', index = False)